### Querying Data: American Community Survey, 5-Year Estimate Detailed Tables

The entry-point for the `acspsuedo` package is the `query` module, particularly the `download()` and `async_download()` functions.

In [1]:
import acspsuedo.query as apq

`acspsuedo` makes use of two models to run queries to the Census Bureau: a standard/synchronous model and an asynchronous concurrent model (built around the built-in `async` library with the `aiohttp.ClientSession` context manager).

URL queries are naturally I/O bound processes, which is why this particular choice of a coroutine execution unit is very much suitable.

A standard query might look something like this.

In [8]:
from acspsuedo.datasets import ACS5
from acspsuedo.fips.states import CA

df = apq.download(
    dataset = ACS5,
    year = 2023,
    tables = 'B25058',
    state = CA,
    tract = '*',
)

df

,NAME,GEO_ID,STATE,COUNTY,TRACT,YEAR,B25058_001E
0,Census Tract 4001; Alameda County; California,1400000US06001400100,06,001,400100,2023,3501.0
1,Census Tract 4002; Alameda County; California,1400000US06001400200,06,001,400200,2023,2795.0
2,Census Tract 4003; Alameda County; California,1400000US06001400300,06,001,400300,2023,2051.0
3,Census Tract 4004; Alameda County; California,1400000US06001400400,06,001,400400,2023,2431.0
4,Census Tract 4005; Alameda County; California,1400000US06001400500,06,001,400500,2023,2042.0
...,...,...,...,...,...,...,...
9124,Census Tract 409.02; Yuba County; California,1400000US06115040902,06,115,040902,2023,2410.0
9125,Census Tract 410.01; Yuba County; California,1400000US06115041001,06,115,041001,2023,905.0
9126,Census Tract 410.02; Yuba County; California,1400000US06115041002,06,115,041002,2023,NaN
9127,Census Tract 411.01; Yuba County; California,1400000US06115041101,06,115,041101,2023,638.0


With the asynchronous approach:

In [9]:
df = await apq.async_download(
    dataset = ACS5,
    year = 2023,
    tables = 'B25058',
    state = CA,
    tract = '*'
)

df

,NAME,GEO_ID,STATE,COUNTY,TRACT,YEAR,B25058_001E
0,Census Tract 4001; Alameda County; California,1400000US06001400100,06,001,400100,2023,3501.0
1,Census Tract 4002; Alameda County; California,1400000US06001400200,06,001,400200,2023,2795.0
2,Census Tract 4003; Alameda County; California,1400000US06001400300,06,001,400300,2023,2051.0
3,Census Tract 4004; Alameda County; California,1400000US06001400400,06,001,400400,2023,2431.0
4,Census Tract 4005; Alameda County; California,1400000US06001400500,06,001,400500,2023,2042.0
...,...,...,...,...,...,...,...
9124,Census Tract 409.02; Yuba County; California,1400000US06115040902,06,115,040902,2023,2410.0
9125,Census Tract 410.01; Yuba County; California,1400000US06115041001,06,115,041001,2023,905.0
9126,Census Tract 410.02; Yuba County; California,1400000US06115041002,06,115,041002,2023,NaN
9127,Census Tract 411.01; Yuba County; California,1400000US06115041101,06,115,041101,2023,638.0


The asynchronous approach may be favorable under the circumstances of:

1. **Querying large amounts of data**, such as an ETL/ELT process or data pipeline


2. **A user-defined API key**, since API key allows for 500+ queries in a single session
    - To learn more about setting up your API key, check out the `API_Key.ipynb` notebook

#### Customizing Query Returns

`acspsuedo.query` supports some customizations for queries, with a default settings for each.

`drop_annotation_variables` (boolean; default True)

The Census Bureau often contains supplementary attribute and margin-of-error
information for data queries. This information may be useful for users interested
in statistical testing and/or data enrichment. By indicating 'False', users can
specify that this information be available in their returned queries.

Cf. [https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html](https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html)

In [3]:
df = apq.download(
    dataset = ACS5,
    year = 2023,
    tables = 'B25058',
    drop_annotation_variables = False,
    state = CA,
    tract = '*',
)

df

,NAME,GEO_ID,STATE,COUNTY,TRACT,YEAR,B25058_001E,B25058_001EA,B25058_001M,B25058_001MA
0,Census Tract 4001; Alameda County; California,1400000US06001400100,06,001,400100,2023,3501.0,"3,500+",NaN,***
1,Census Tract 4002; Alameda County; California,1400000US06001400200,06,001,400200,2023,2795.0,NaN,640.0,NaN
2,Census Tract 4003; Alameda County; California,1400000US06001400300,06,001,400300,2023,2051.0,NaN,370.0,NaN
3,Census Tract 4004; Alameda County; California,1400000US06001400400,06,001,400400,2023,2431.0,NaN,245.0,NaN
4,Census Tract 4005; Alameda County; California,1400000US06001400500,06,001,400500,2023,2042.0,NaN,222.0,NaN
...,...,...,...,...,...,...,...,...,...,...
9124,Census Tract 409.02; Yuba County; California,1400000US06115040902,06,115,040902,2023,2410.0,NaN,134.0,NaN
9125,Census Tract 410.01; Yuba County; California,1400000US06115041001,06,115,041001,2023,905.0,NaN,413.0,NaN
9126,Census Tract 410.02; Yuba County; California,1400000US06115041002,06,115,041002,2023,NaN,-,NaN,**
9127,Census Tract 411.01; Yuba County; California,1400000US06115041101,06,115,041101,2023,638.0,NaN,70.0,NaN


`convert_to_na` (boolean; default True)

The Census Bureau may indicate particular records with unique integer and/or non-integer values
to designate some particular characteristics of queried tables, such as insufficient sample sizes
or estimates falling outside the range of an open-ended distribution. By indicating 'False',
users are indicating that none of these special values be replaced with `numpy.nan` values.

Cf. [https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html](https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html)

In [11]:
df = apq.download(
    dataset = ACS5,
    year = 2023,
    tables = 'B25058',
    convert_to_na = False,
    state = CA,
    tract = '*',
)

df

,NAME,GEO_ID,STATE,COUNTY,TRACT,YEAR,B25058_001E
0,Census Tract 4001; Alameda County; California,1400000US06001400100,06,001,400100,2023,3501
1,Census Tract 4002; Alameda County; California,1400000US06001400200,06,001,400200,2023,2795
2,Census Tract 4003; Alameda County; California,1400000US06001400300,06,001,400300,2023,2051
3,Census Tract 4004; Alameda County; California,1400000US06001400400,06,001,400400,2023,2431
4,Census Tract 4005; Alameda County; California,1400000US06001400500,06,001,400500,2023,2042
...,...,...,...,...,...,...,...
9124,Census Tract 409.02; Yuba County; California,1400000US06115040902,06,115,040902,2023,2410
9125,Census Tract 410.01; Yuba County; California,1400000US06115041001,06,115,041001,2023,905
9126,Census Tract 410.02; Yuba County; California,1400000US06115041002,06,115,041002,2023,-666666666
9127,Census Tract 411.01; Yuba County; California,1400000US06115041101,06,115,041101,2023,638


*For `async_download` only*

<br>

`retry_rate` (integer; default 30)

For the case of larger queries, server-blocking may hamper how many request attempts can be successfully made with the desired return output. Thus, users may be interested in specifying a custom amount of request attempts to be made before failing by setting the `retry_rate` parameter.

<br>

`timeout_rate` (float or integer; default 0.1)

Likewise, users may be interested in pacing request attempts in the occasion that there is server-based blocking for large queries. By setting the `timeout_rate`, users are effectively specifying how many seconds should be waited in between request attempts (both successful and failed). The default is 0.1 seconds.


In [10]:
df = await apq.async_download(
    dataset = ACS5,
    year = 2023,
    tables = 'B25058',
    retry_rate = 10,
    timeout_rate = 1,
    state = CA,
    tract = '*'
)

df

,NAME,GEO_ID,STATE,COUNTY,TRACT,YEAR,B25058_001E
0,Census Tract 4001; Alameda County; California,1400000US06001400100,06,001,400100,2023,3501.0
1,Census Tract 4002; Alameda County; California,1400000US06001400200,06,001,400200,2023,2795.0
2,Census Tract 4003; Alameda County; California,1400000US06001400300,06,001,400300,2023,2051.0
3,Census Tract 4004; Alameda County; California,1400000US06001400400,06,001,400400,2023,2431.0
4,Census Tract 4005; Alameda County; California,1400000US06001400500,06,001,400500,2023,2042.0
...,...,...,...,...,...,...,...
9124,Census Tract 409.02; Yuba County; California,1400000US06115040902,06,115,040902,2023,2410.0
9125,Census Tract 410.01; Yuba County; California,1400000US06115041001,06,115,041001,2023,905.0
9126,Census Tract 410.02; Yuba County; California,1400000US06115041002,06,115,041002,2023,NaN
9127,Census Tract 411.01; Yuba County; California,1400000US06115041101,06,115,041101,2023,638.0
